# ENERGICAL — RFM Clustering & Customer Segmentation
### DataFest 2026 | The outliers | June 2026

## 0. Imports & Configuration

In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')


## 1. Load cleaned data

We load already cleaned in 01_eda_cleaning.ipynb.
This notebook focuses on customer segmentation via RFM clustering.


In [8]:
df = pd.read_csv(r"C:\Users\PCPRODZ\Downloads\df_clean.csv")
df['date_commande'] = pd.to_datetime(df['date_commande'])

print("=" * 50)
print("SHAPE:", df.shape)
print("=" * 50)
print(df.head())


SHAPE: (16617, 16)
   id_transaction  id_commande_anon  id_client type_client nouveau_ou_fidele  \
0               1             10872  CLT_02355         B2C         Returning   
1               2             12712  CLT_00422         B2C               New   
2               3             17642  CLT_03812         B2C         Returning   
3               4             16497  CLT_02886         B2C         Returning   
4               5             17671  CLT_00452         B2C               New   

  date_commande     wilaya categorie_produit  \
0    2022-01-03   Laghouat       Électricité   
1    2022-12-08      Adrar       Électricité   
2    2024-06-10       Mila         Sanitaire   
3    2024-01-31  Boumerdès       Électricité   
4    2024-06-13      Alger         Sanitaire   

                                             produit  quantite  montant_da  \
0                                Electrode détecteur       3.0      826.05   
1            Interrupteur Simple Allumage DIAA Métal   

## 2. RFM Analysis

In [3]:
snapshot_date = df['date_commande'].max() + pd.Timedelta(days=1)

rfm = df.groupby('id_client').agg(
    recence   = ('date_commande', lambda x: (snapshot_date - x.max()).days),
    frequence = ('id_transaction', 'count'),
    montant   = ('montant_da', 'sum')
).reset_index()

print("\n" + "=" * 50)
print("RFM — Aperçu")
print("=" * 50)
print(rfm.describe())


RFM — Aperçu
           recence    frequence       montant
count  4112.000000  4112.000000  4.112000e+03
mean    453.137403     4.041099  5.479697e+04
std     308.343757    31.021834  1.158773e+06
min       1.000000     1.000000  5.744000e+01
25%     185.750000     1.000000  3.920560e+03
50%     409.000000     1.000000  1.382681e+04
75%     689.250000     3.000000  2.917626e+04
max    1096.000000  1850.000000  7.336871e+07


### 📊 RFM baseline
- **Avg recency**: 453 days — customers haven't bought in ~1.2 years
- **Avg frequency**: 4 orders per customer
- **Avg monetary**: 54,798 DZD

→ Most customers are one-time/low-frequency buyers.

## 3. Clustering (K-Means)

In [4]:

rfm = rfm[rfm['frequence'] < 100]  

scaler     = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['recence', 'frequence', 'montant']])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
rfm['segment'] = kmeans.fit_predict(rfm_scaled)

segment_summary = rfm.groupby('segment').agg(
    recence_moy   = ('recence', 'mean'),
    frequence_moy = ('frequence', 'mean'),
    montant_moy   = ('montant', 'mean'),
    nb_clients    = ('id_client', 'count')
).reset_index()

print("\n" + "=" * 50)
print("SEGMENTS CLIENTS")
print("=" * 50)
print(segment_summary)

def label_segment(row):
    if row['frequence_moy'] > 10:
        return 'Champions 🟢'
    elif row['recence_moy'] < 300:
        return 'A risque 🟡'
    else:
        return 'Perdus 🔴'

segment_summary['label'] = segment_summary.apply(label_segment, axis=1)
print(segment_summary[['segment', 'label', 'nb_clients']])

label_map = segment_summary.set_index('segment')['label'].to_dict()
rfm['label'] = rfm['segment'].map(label_map)

print("\n✅ RFM terminé — prêt pour le dashboard!")


SEGMENTS CLIENTS
   segment  recence_moy  frequence_moy    montant_moy  nb_clients
0        0   758.414496       2.397508   29239.161246        1766
1        1   226.805446       2.952130   27162.708639        2277
2        2   100.730159      43.619048  369181.214127          63
   segment        label  nb_clients
0        0     Perdus 🔴        1766
1        1   A risque 🟡        2277
2        2  Champions 🟢          63

✅ RFM terminé — prêt pour le dashboard!


### 📊 Segment breakdown
- **Champions (63)**: High frequency (43.6 orders), recent (101 days ago)
- **At-risk (2,277)**: Low frequency, purchased 227 days ago
- **Dormant (1,766)**: Haven't bought in 758 days — churn risk

→ **Business number**: 2,277 at-risk clients = Y DZD revenue threatened

## 4. Visualisation RFM

In [5]:
fig = px.scatter(
    rfm,
    x='recence',
    y='frequence',
    size='montant',
    color='label',
    hover_data=['id_client', 'montant'],
    title='Segmentation Clients RFM — Energical',
    labels={
        'recence'  : 'Récence (jours)',
        'frequence': 'Fréquence (commandes)',
        'montant'  : 'Montant (DA)',
        'label'    : 'Segment'
    },
    color_discrete_map={
        'Champions 🟢': 'green',
        'A risque 🟡' : 'orange',
        'Perdus 🔴'   : 'red'
    }
)
fig.show()


## 5. Export for the dashboard

In [ ]:

rfm.to_csv('rfm_clients.csv', index=False)

print("\n📁 rfm_clients.csv")


📁 Fichiers exportés: df_clean.csv | rfm_clients.csv
